# 02 · Train Encoders — Multi-model comparison

This notebook is the **orchestrator**: it imports all business logic from `src/`
and loops over models × augmentation strategies × loss functions.

All results are saved to `results/` as JSON and can be compared at the end
(or in `03_compare_results.ipynb`).

---
### Quick-start
1. Adjust `MODELS_TO_RUN`, `AUGMENTATIONS`, `LOSSES` in the **Configuration** cell.
2. Run all cells top to bottom.
3. Results appear in `results/` and in the final comparison table.

## 0 · Setup (Colab only)

In [ ]:
# ── Google Colab setup ────────────────────────────────────────────────────────
# This cell is a no-op when running locally.
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Drive to persist results across sessions (optional)
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone the project repo if running directly from Colab
    # !git clone https://github.com/YOUR_USER/YOUR_REPO.git
    # %cd YOUR_REPO

    # Install dependencies
    !pip install -q transformers datasets scikit-learn torch nltk
    !pip uninstall -y torchvision # Avoid potential version conflicts with Colab's pre-installed version
    print('Colab dependencies installed.')
else:
    print('Running locally — skipping Colab setup.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab dependencies installed.


In [2]:
# Add project root to sys.path so `src/` is importable from any working directory
import sys
from pathlib import Path

# Move the project to `progettoLLM/` folder in your Drive and update the path below if needed
PROJECT_ROOT = Path('/content/drive/MyDrive/progettoLLM/CLARITY').resolve()   # notebooks/ is one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

Project root: /content/drive/MyDrive/progettoLLM/CLARITY


## 1 · Configuration

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EDIT THIS CELL to control what gets trained.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from config.model_configs import MODEL_CONFIGS
from src.data.augmentation import list_augmentations

# ── Models to benchmark ───────────────────────────────────────────────────────
# Comment out any model you don't want to run.
MODELS= [
    'distilbert-base-uncased',
    #'longformer-base',              # original notebook baseline
    #'roberta-base',                 # paper baseline
    # 'xlnet-base',                 # paper baseline — slower, enable if needed
    # 'longformer-large',           # needs >20 GB VRAM
    #'bert-base-uncased',          # reference

]

# ── Balancing strategies ───────────────────────────────────────────────────────
# Available: 'none', 'semantic_downsampling', 'Paraphrase Upsampling', 'Smart Resampling'
# Note: it may takes a while, especially upsampling strategy
BALANCING_STRATEGIES = [
     #'none',             # control: no balancing
    #'semantic_downsampling',  # downsample majority classes based on semantic similarity
    # 'paraphrase_upsampling',  # augment minority classes with paraphrases
    'smart_resampling',  # combine downsampling and upsampling based on class difficulty
]

# ── Augmentation strategies ───────────────────────────────────────────────────
# Available: print(list_augmentations())
AUGMENTATIONS = [
     'none',                  # control: no augmentation
    # 'synonym_replacement',   # EDA-style synonym swap on minority classes
    # 'oversampling',          # duplicate minority examples to balance
    # 'random_deletion',
    # 'random_swap',
    # 'back_translation',     # slow — needs GPU and internet
    # 'length_category',      # adds a length-based feature (ablation)
    #'tone_analysis',          # adds a tone-based feature (ablation)
]

# ── Loss functions ────────────────────────────────────────────────────────────
# Available: 'focal', 'weighted_ce', 'label_smoothing', 'ce'
LOSSES = [
    #'focal',         # original notebook loss
    # 'weighted_ce', # ablation: class-weighted CE without focal modulation
    'ce',          # ablation: vanilla cross-entropy (no weighting)
    # 'label_smoothing',  # ablation: CE with label smoothing instead of class weights
]

# ── Shared hyperparameters ────────────────────────────────────────────────────
NUM_EPOCHS    = 5
WEIGHT_DECAY  = 0.01
FOCAL_GAMMA   = 2.0
SEED          = 42

# ── Task setting ──────────────────────────────────────────────────────────────
# 'evasion'  → 9-class fine-grained task  (used in the original notebook)
# 'clarity'  → 3-class high-level task    (Clear Reply / Ambivalent / Clear Non-Reply)
TASK = 'clarity'

print('Models     :', MODELS)
print('Balancing  :', BALANCING_STRATEGIES)
print('Augment.   :', AUGMENTATIONS)
print('Losses     :', LOSSES)
print('Total runs :', len(MODELS) * len(AUGMENTATIONS) * len(LOSSES))

Models     : ['distilbert-base-uncased']
Balancing  : ['smart_resampling']
Augment.   : ['none']
Losses     : ['ce']
Total runs : 1


## 2 · Load dataset (once — shared by all runs)

In [4]:
from src.data.dataset_loader import load_and_split_dataset, EVASION_TO_CLARITY

print('Loading ailsntua/QEvasion ...')
raw_ds = load_and_split_dataset(seed=SEED, verbose=True)
raw_ds

Loading ailsntua/QEvasion ...
  train       :  2758 samples
  validation  :   690 samples
  test        :   308 samples


DatasetDict({
    train: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 2758
    })
    validation: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 690
    })
    test: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'anno

## 3 · Build label maps (once)

In [5]:
from src.data.label_utils import build_label_maps, compute_alpha_weights, apply_labels

# Select the label column based on the task
import src.data.label_utils as lu
if TASK == 'clarity':
    lu.LABEL_COLUMN = 'clarity_label'   # switch to 3-class task
else:
    lu.LABEL_COLUMN = 'evasion_label'   # default 9-class task

label2id, id2label = build_label_maps(raw_ds)
num_classes = len(label2id)

print(f'Task          : {TASK}')
print(f'Num classes   : {num_classes}')
print(f'Label mapping : {label2id}')

# Apply integer labels to all splits
labeled_ds = apply_labels(raw_ds, label2id, verbose=True)

Task          : evasion
Num classes   : 9
Label mapping : {'Claims ignorance': 0, 'Clarification': 1, 'Declining to answer': 2, 'Deflection': 3, 'Dodging': 4, 'Explicit': 5, 'General': 6, 'Implicit': 7, 'Partial/half-answer': 8}

Alpha weights (inverse frequency):
  Claims ignorance          : 3.1270
  Clarification             : 4.2562
  Declining to answer       : 2.5970
  Deflection                : 0.9982
  Dodging                   : 0.5502
  Explicit                  : 0.3567
  General                   : 0.9885
  Implicit                  : 0.8260
  Partial/half-answer       : 4.6431


Map:   0%|          | 0/308 [00:00<?, ? examples/s]


[train] label distribution:
  Claims ignorance          :   98
  Clarification             :   72
  Declining to answer       :  118
  Deflection                :  307
  Dodging                   :  557
  Explicit                  :  859
  General                   :  310
  Implicit                  :  371
  Partial/half-answer       :   66

[validation] label distribution:
  Claims ignorance          :   21
  Clarification             :   20
  Declining to answer       :   27
  Deflection                :   74
  Dodging                   :  149
  Explicit                  :  193
  General                   :   76
  Implicit                  :  117
  Partial/half-answer       :   13

[test] label distribution:
  Claims ignorance          :    8
  Clarification             :    4
  Declining to answer       :   11
  Deflection                :   20
  Dodging                   :   50
  Explicit                  :   79
  General                   :   65
  Implicit                  :   67

## 4 · Main training loop

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, EarlyStoppingCallback

from config.model_configs import get_model_config
from src.data.augmentation import get_augmentation_fn
from src.training.metrics import build_compute_metrics_fn, compute_detailed_report
from src.training.trainers import get_trainer
from src.utils.env_utils import get_training_args
from src.utils.results_utils import save_results


def free_memory():
    """Release GPU/CPU memory between runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def tokenize_for_model(ds, tokenizer, model_cfg: dict):
    """Tokenize the dataset for a specific model configuration."""
    max_length = model_cfg['max_length']
    use_token_type_ids = model_cfg.get('token_type_ids', False)

    def tokenize_fn(examples):
        modified_answers = []

        # Verifichiamo se le feature esistono in questo split del dataset
        has_tone = 'tone' in examples
        has_length = 'length_category' in examples


        for i in range(len(examples['interview_answer'])):
            ans = str(examples['interview_answer'][i])

             # Estraiamo i valori solo se la colonna esiste, altrimenti stringa vuota
            t = str(examples['tone'][i]) if has_tone and examples['tone'][i] else ""
            l = str(examples['length_category'][i]) if has_length and examples['length_category'][i] else ""

            prefix_parts = []
            if t:
                prefix_parts.append(f"Tone: {t}")
            if l:
                prefix_parts.append(f"Length: {l}")

            if prefix_parts:
                prefix = " | ".join(prefix_parts) + " | Answer: "
            else:
                prefix = ""

            modified_answers.append(prefix + ans)

        return tokenizer(
            examples['question'],
            modified_answers, # Passiamo le risposte modificate col prefisso
            padding='max_length',
            truncation=True,
            max_length=max_length,
        )

    tokenized = ds.map(tokenize_fn, batched=True)

    # Keep only the columns the model actually needs
    columns_to_keep = ['input_ids', 'attention_mask', 'label']
    if use_token_type_ids:
        columns_to_keep.append('token_type_ids')

    # Some models (e.g. Longformer) also produce global_attention_mask
    if 'global_attention_mask' in tokenized['train'].column_names:
        columns_to_keep.append('global_attention_mask')

    tokenized.set_format(type='torch', columns=columns_to_keep)
    return tokenized


aug_tag_name = "_".join(AUGMENTATIONS)
bal_tag_name = "_".join(BALANCING_STRATEGIES)

# ── Outer loop ────────────────────────────────────────────────────────────────
all_run_results = []
master_ds = labeled_ds
for bal in BALANCING_STRATEGIES:
    print(f"Applying Resampling: {bal}")
    master_ds = get_augmentation_fn(bal)(master_ds, label2id, seed=SEED)
    free_memory()

# B. NLP Augmentations
for aug in AUGMENTATIONS:
    print(f"Applying Augmentation: {aug}")
    master_ds = get_augmentation_fn(aug)(master_ds, label2id, seed=SEED)
    free_memory()

print(f"\nFinal Train Size: {len(master_ds['train'])}")

# Compute inverse-frequency weights (for FocalLoss / WeightedCE)
alpha_tensor = compute_alpha_weights(master_ds, label2id)
print('\nAlpha weights (inverse frequency):')
for i, w in enumerate(alpha_tensor.tolist()):
    print(f'  {id2label[i]:<25} : {w:.4f}')

for model_key in MODELS:
    model_cfg = get_model_config(model_key)
    model_id  = model_cfg['model_id']

    print(f'\n{"="*70}')
    print(f'MODEL : {model_key}  ({model_id})')
    print(f'{"="*70}')

    # Load tokenizer once per model
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenized_ds = tokenize_for_model(master_ds, tokenizer, model_cfg)


    for loss_name in LOSSES:
        print(f'\n    ── Loss: {loss_name} ──')

        run_tag = f'{model_key} / {loss_name}'

        # ── Build model (fresh weights each run) ──────────────────────────
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=num_classes,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )

        # ── Build TrainingArguments ───────────────────────────────────────
        training_args = get_training_args(
            model_key=model_key,
            model_cfg=model_cfg,
            bal_name=bal_tag_name,
            aug_name=aug_tag_name,
            loss_name=loss_name,
            num_epochs=NUM_EPOCHS,
            weight_decay=WEIGHT_DECAY,
        )

        # Configure dynamic metrics computing depending on task
        eval_mapping = EVASION_TO_CLARITY if TASK == 'evasion' else None
        custom_compute_metrics = build_compute_metrics_fn(
            id2label=id2label,
            evasion_to_clarity=eval_mapping
        )

        # ── Build Trainer ─────────────────────────────────────────────────
        trainer = get_trainer(
            loss_name=loss_name,
            model=model,
            training_args=training_args,
            train_dataset=tokenized_ds['train'],
            eval_dataset=tokenized_ds['validation'],
            compute_metrics=custom_compute_metrics,
            alpha=alpha_tensor,
            gamma=FOCAL_GAMMA,
            num_classes=num_classes,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        # ── Train ─────────────────────────────────────────────────────────
        print(f'    Starting training for: {run_tag}')
        trainer.train()

        # ── Evaluate on VALIDATION set ────────────────────────────────────
        val_metrics = trainer.evaluate()
        print(f'    Validation metrics: {val_metrics}')

        # ── Evaluate on TEST set ──────────────────────────────────────────
        test_pred = trainer.predict(tokenized_ds['test'])
        test_metrics = test_pred.metrics
        print(f'    Test metrics: {test_metrics}')

        # Detailed per-class report
        report = compute_detailed_report(
            test_pred.predictions,
            test_pred.label_ids,
            id2label,
            evasion_to_clarity=eval_mapping,
        )
        print(f'\n    Per-class report:\n{report}')

        # ── Save results ──────────────────────────────────────────────────
        metrics_dict = {
            'val_macro_f1':      val_metrics.get('eval_macro_f1', 0),
            'val_weighted_f1':   val_metrics.get('eval_weighted_f1', 0),
            'val_accuracy':      val_metrics.get('eval_accuracy', 0),
            'test_macro_f1':     test_metrics.get('test_macro_f1', 0),
            'test_weighted_f1':  test_metrics.get('test_weighted_f1', 0),
            'test_accuracy':     test_metrics.get('test_accuracy', 0),
        }

        if eval_mapping:
            metrics_dict.update({
                'val_clarity_macro_f1':      val_metrics.get('eval_clarity_macro_f1', 0),
                'val_clarity_accuracy':      val_metrics.get('eval_clarity_accuracy', 0),
                'test_clarity_macro_f1':     test_metrics.get('test_clarity_macro_f1', 0),
                'test_clarity_accuracy':     test_metrics.get('test_clarity_accuracy', 0),
            })

        save_results(
            model_key=model_key,
            bal_name=bal_tag_name,
            aug_name=aug_tag_name,
            loss_name=loss_name,
            metrics=metrics_dict,
            extra={
                'model_id':    model_id,
                'max_length':  model_cfg['max_length'],
                'num_epochs':  NUM_EPOCHS,
                'focal_gamma': FOCAL_GAMMA,
                'task':        TASK,
                'train_size':  len(master_ds['train']),
            },
        )
        all_run_results.append(run_tag)

        # ── Clean up before next run ──────────────────────────────────────
        del model, trainer
        free_memory()

    del tokenized_ds, tokenizer
    free_memory()

print('\n\nAll runs completed:')
for r in all_run_results:
    print(' ✓', r)

Applying Resampling: smart_resampling

[smart_resampling] Strategy: 'soft'
[smart_resampling] Target distribution:
  Declining to answer: 118 → 236
  Implicit: 371 → 644
  General: 310 → 620
  Explicit: 859 → 644
  Dodging: 557 → 644
  Deflection: 307 → 614
  Clarification: 72 → 144
  Claims ignorance: 98 → 196
  Partial/half-answer: 66 → 132

[semantic_downsampling] Initializing 'all-MiniLM-L6-v2' ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Filter:   0%|          | 0/2758 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2758 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


    Validation metrics: {'eval_loss': 1.8319896459579468, 'eval_macro_f1': 0.34013903086793007, 'eval_weighted_f1': 0.3548074929413754, 'eval_accuracy': 0.3536231884057971, 'eval_clarity_macro_f1': 0.5740336828993896, 'eval_clarity_weighted_f1': 0.643860763535473, 'eval_clarity_accuracy': 0.6536231884057971, 'eval_runtime': 2.7591, 'eval_samples_per_second': 250.081, 'eval_steps_per_second': 3.987, 'epoch': 5.0}
    Test metrics: {'test_loss': 1.9536898136138916, 'test_macro_f1': 0.2935710600287167, 'test_weighted_f1': 0.23545136509206338, 'test_accuracy': 0.2435064935064935, 'test_clarity_macro_f1': 0.5067886081044586, 'test_clarity_weighted_f1': 0.5942886040428245, 'test_clarity_accuracy': 0.5876623376623377, 'test_runtime': 1.24, 'test_samples_per_second': 248.395, 'test_steps_per_second': 4.032}

    Per-class report:
                     precision    recall  f1-score   support

   Claims ignorance       0.46      0.75      0.57         8
      Clarification       1.00      0.75   

## 5 · Quick comparison table

In [7]:
from src.utils.results_utils import compare_results, print_comparison_table

df = compare_results(
    metrics_to_show=['test_macro_f1', 'test_weighted_f1', 'test_accuracy',
                     'val_macro_f1']
)
df

,model,augmentation,loss,timestamp,test_macro_f1,test_weighted_f1,test_accuracy,val_macro_f1
0,bert-base-uncased,none,ce,2026-04-28T11:30:44,0.5787,0.6908,0.7078,0.6287
1,bert-base-uncased,none,focal,2026-04-28T10:29:59,0.6014,0.6771,0.6721,0.6176
2,bert-base-uncased,tone_analysis,ce,2026-04-26T13:16:06,0.5946,0.6732,0.6688,0.6155
3,deberta-v3-base,tone_analysis,focal,2026-05-06T09:48:37,0.2672,0.5361,0.6688,0.2560
4,distilbert-base-uncased,none,ce,2026-05-28T16:14:03,0.2872,0.2134,0.2695,0.3060
5,distilbert-base-uncased,none,focal,2026-04-28T10:05:40,0.5409,0.6119,0.6006,0.5681
6,distilbert-base-uncased,none,label_smoothing,2026-05-07T17:02:41,0.5461,0.6543,0.6786,0.6093
7,distilbert-base-uncased,smart_resampling,ce,2026-05-28T16:50:12,0.2936,0.2355,0.2435,0.3401
8,distilbert-base-uncased,smart_resampling,focal,2026-05-12T15:01:21,0.5063,0.6219,0.6136,0.5739
9,distilbert-base-uncased,smart_resampling,label_smoothing,2026-05-12T15:06:21,0.5184,0.5852,0.5714,0.5963


In [8]:
# Rich table (if rich is installed, otherwise plain print)
print_comparison_table()

                                     Experiment Results (sorted by Macro F1 ↓)                                     
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ model             ┃ augmentation     ┃ loss            ┃ timestamp          ┃ macro_f1 ┃ weighted_f1 ┃ accuracy ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ bert-base-uncased │ none             │ ce              │ 2026-04-28T11:30:… │      nan │         nan │      nan │
│ bert-base-uncased │ none             │ focal           │ 2026-04-28T10:29:… │      nan │         nan │      nan │
│ bert-base-uncased │ tone_analysis    │ ce              │ 2026-04-26T13:16:… │      nan │         nan │      nan │
│ deberta-v3-base   │ tone_analysis    │ focal           │ 2026-05-06T09:48:… │      nan │         nan │      nan │
│ distilbert-base-… │ none             │ ce              │ 2026-05-28T16:14:… │      nan │         nan │      nan │
│ distilbert-base-… │ none             │ focal           │ 2026-04-28T10:05:… │      nan │         nan │      nan │
│ distilbert-base-… │ none             │ label_smoothing │ 2026-05-07T17:02:… │      nan │         nan │      nan │
│ distilbert-base-… │ smart_resampling │ ce              │ 2026-05-28T16:50:… │      nan │         nan │      nan │
│ distilbert-base-… │ smart_resampling │ focal           │ 2026-05-12T15:01:… │      nan │         nan │      nan │
│ distilbert-base-… │ smart_resampling │ label_smoothing │ 2026-05-12T15:06:… │      nan │         nan │      nan │
│ distilbert-base-… │ tone_analysis    │ ce              │ 2026-04-26T11:41:… │      nan │         nan │      nan │
│ roberta-base      │ none             │ ce              │ 2026-05-22T07:57:… │      nan │         nan │      nan │
│ roberta-base      │ none             │ focal           │ 2026-04-28T10:18:… │      nan │         nan │      nan │
│ roberta-base      │ none             │ label_smoothing │ 2026-05-07T17:13:… │      nan │         nan │      nan │
│ roberta-base      │ smart_resampling │ ce              │ 2026-05-12T15:56:… │      nan │         nan │      nan │
│ roberta-base      │ smart_resampling │ focal           │ 2026-05-12T15:16:… │      nan │         nan │      nan │
│ roberta-base      │ smart_resampling │ label_smoothing │ 2026-05-12T15:27:… │      nan │         nan │      nan │
│ roberta-base      │ tone_analysis    │ ce              │ 2026-04-26T13:05:… │      nan │         nan │      nan │
└───────────────────┴──────────────────┴─────────────────┴────────────────────┴──────────┴─────────────┴──────────┘